# Unità 2 — BB84 con Eve: attacco intercept-resend

In questo notebook introduciamo Eve nel protocollo BB84. Eve intercetta il qubit, lo misura in una base scelta casualmente e poi reinvia a Bob un nuovo qubit coerente con il risultato della sua misura.

## Setup e import

Prepariamo il percorso del progetto in modo da poter importare le funzioni dalla cartella `src`.

In [ ]:
from pathlib import Path
import sys

current_path = Path.cwd()

if (current_path / "src" / "bb84.py").exists():
    project_root = current_path
elif (current_path.parent / "src" / "bb84.py").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError("Non trovo la cartella src del progetto.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

In [ ]:
import importlib
import pandas as pd

import bb84
import attacks

importlib.reload(bb84)
importlib.reload(attacks)

from attacks import (
    choose_eve_basis,
    eve_measure,
    eve_intercept_resend,
)

from bb84 import (
    run_bb84_round_with_eve,
    run_bb84_protocol_with_eve,
    sift_keys,
    compute_qber,
)

## Richiamo del caso ideale

Nel notebook precedente abbiamo simulato BB84 ideale. In assenza di Eve e rumore, dopo il sifting Alice e Bob ottengono la stessa chiave e il QBER è nullo.

In [ ]:
results_no_eve = run_bb84_protocol_with_eve(
    n_rounds=50,
    intercept_probability=0.0,
    seed=123,
)

alice_key_no_eve, bob_key_no_eve = sift_keys(results_no_eve)
qber_no_eve = compute_qber(alice_key_no_eve, bob_key_no_eve)

print("Lunghezza della chiave sifted senza Eve:", len(alice_key_no_eve))
print("QBER senza Eve:", qber_no_eve)

## Scelta della base di Eve

Eve non conosce la base scelta da Alice. Per questo sceglie casualmente una base tra Z e X.

In [ ]:
eve_basis = choose_eve_basis(seed=1)

print("Base scelta da Eve:", eve_basis)

## Misura di Eve

Eve misura il qubit nella propria base. Se la base di Eve coincide con quella di Alice, il bit viene recuperato correttamente. Se la base è diversa, la misura introduce casualità.

In [ ]:
eve_bit_same_basis = eve_measure(
    alice_bit=1,
    alice_basis="Z",
    eve_basis="Z",
)
print("Bit misurato da Eve nella stessa base:", eve_bit_same_basis)

results_eve_wrong_basis = []

for i in range(10):
    eve_bit = eve_measure(
        alice_bit=0,
        alice_basis="X",
        eve_basis="Z",
    )
    results_eve_wrong_basis.append(eve_bit)

print("Misure di Eve con base diversa:", results_eve_wrong_basis)

## Intercept-resend

Nell'attacco intercept-resend, Eve misura il qubit e poi reinvia a Bob un nuovo qubit preparato nella base e con il bit ottenuti dalla sua misura.

In [ ]:
eve_result = eve_intercept_resend(
    alice_bit=1,
    alice_basis="X",
    eve_basis="Z",
)

print(eve_result)

## Singolo round BB84 con Eve

Ora simuliamo un singolo round completo. Se Eve intercetta, Bob non riceve direttamente lo stato di Alice, ma quello ricostruito da Eve.

In [ ]:
round_with_eve = run_bb84_round_with_eve(
    alice_bit=1,
    alice_basis="X",
    bob_basis="X",
    intercept_probability=1.0,
    seed=10,
)

print(round_with_eve)

## Protocollo multi-round con Eve

Ripetiamo il protocollo per più round. Per ogni round salviamo le scelte di Alice e Bob, il bit misurato da Bob e le informazioni essenziali sull'intercettazione di Eve.

In [ ]:
results_with_eve = run_bb84_protocol_with_eve(
    n_rounds=50,
    intercept_probability=1.0,
    seed=123,
)

for i in range(5):
    print(results_with_eve[i])

## Sifting con Eve

Anche in presenza di Eve, Alice e Bob applicano il sifting: mantengono solo i round in cui hanno scelto la stessa base.

In [ ]:
alice_key_with_eve, bob_key_with_eve = sift_keys(results_with_eve)

print("Chiave di Alice con Eve:", alice_key_with_eve)
print("Chiave di Bob con Eve:", bob_key_with_eve)
print("Lunghezza della chiave sifted con Eve:", len(alice_key_with_eve))

## QBER con Eve

Il QBER misura la frazione di errori tra la chiave di Alice e la chiave di Bob dopo il sifting. Con un attacco intercept-resend, ci si aspetta un QBER maggiore rispetto al caso ideale.

In [ ]:
qber_with_eve = compute_qber(alice_key_with_eve, bob_key_with_eve)

print("QBER con Eve:", qber_with_eve)

## Confronto essenziale

Confrontiamo il QBER senza Eve e con Eve sempre attiva. Il primo caso rappresenta il riferimento ideale; il secondo mostra l'effetto dell'intercettazione.

In [ ]:
print("QBER senza Eve:", qber_no_eve)
print("QBER con Eve:", qber_with_eve)

if qber_with_eve > qber_no_eve:
    print("L'intercettazione di Eve ha aumentato il QBER in questa simulazione.")
else:
    print("In questa simulazione il QBER non è aumentato. Con pochi round possono esserci fluttuazioni statistiche.")

## Rappresentazione tabellare

La simulazione restituisce una lista di dizionari. Per visualizzare meglio i risultati possiamo convertirla in una tabella pandas. Questa fase è separata dalla logica del protocollo.

In [ ]:
df_with_eve = pd.DataFrame(results_with_eve)
df_with_eve.head(10)

## Commento finale

Nel caso ideale il QBER è nullo. Quando Eve applica un attacco intercept-resend, la misura in una base potenzialmente diversa da quella di Alice può disturbare lo stato quantistico. Questo disturbo può produrre errori nella chiave sifted e quindi aumentare il QBER. Nei prossimi passaggi il confronto sarà reso più sistematico variando la probabilità di intercettazione.